In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Commit 1: Model Performance Evaluation
# feat(task5): full evaluation metrics for distributed ML models
#
# Coursework Requirement:
# Compare the performance of all trained classifiers using multiple
# evaluation metrics and export the results for reporting.
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import time
import pandas as pd

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA_Task5")
    .master("local[*]")
    .config("spark.driver.memory", "10g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# -----------------------------------------------------------------------------
# Project Paths
# -----------------------------------------------------------------------------

BASE = "/content/drive/MyDrive/7006SCN"

PROCESSED = f"{BASE}/data/processed"
MODELS = f"{BASE}/models"
OUTPUT = f"{BASE}/outputs"

# -----------------------------------------------------------------------------
# Load HMDA Dataset
# -----------------------------------------------------------------------------

from pyspark.ml import PipelineModel

print("=" * 80)
print("LOADING HMDA DATASET")
print("=" * 80)

df_clean = spark.read.parquet(
    f"{PROCESSED}/hmda_clean.parquet"
)

print(f"Rows    : {df_clean.count():,}")
print(f"Columns : {len(df_clean.columns)}")

# -----------------------------------------------------------------------------
# Load Preprocessing Pipeline
# -----------------------------------------------------------------------------

pipeline_model = PipelineModel.load(
    f"{MODELS}/preprocessing_pipeline"
)

print("Preprocessing Pipeline Loaded ✓")

# -----------------------------------------------------------------------------
# Recreate Task 3 Test Dataset
# -----------------------------------------------------------------------------

print("\nCreating Evaluation Dataset...")

_, test_clean = df_clean.randomSplit(
    [0.8, 0.2],
    seed=7006
)

test_clean = test_clean.sample(
    withReplacement=False,
    fraction=0.10,
    seed=7006
)

print("Applying preprocessing pipeline...")

test = pipeline_model.transform(test_clean)

test.cache()

test_rows = test.count()

print(f"Testing Rows : {test_rows:,}")

# -----------------------------------------------------------------------------
# Load Optimised Models
# -----------------------------------------------------------------------------

from pyspark.ml.classification import (
    LogisticRegressionModel,
    DecisionTreeClassificationModel,
    RandomForestClassificationModel,
    GBTClassificationModel
)

models = {

    "Logistic Regression":
        LogisticRegressionModel.load(f"{MODELS}/lr_cv_best"),

    "Decision Tree":
        DecisionTreeClassificationModel.load(f"{MODELS}/dt_best"),

    "Random Forest":
        RandomForestClassificationModel.load(f"{MODELS}/rf_best"),

    "Gradient Boosted Trees":
        GBTClassificationModel.load(f"{MODELS}/gbt_best")
}

print("\nLoaded Models")

for model in models:
    print("✓", model)

# -----------------------------------------------------------------------------
# Evaluation Metrics
# -----------------------------------------------------------------------------

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

ev_acc = MulticlassClassificationEvaluator(
    labelCol="loan_approval",
    predictionCol="prediction",
    metricName="accuracy"
)

ev_f1 = MulticlassClassificationEvaluator(
    labelCol="loan_approval",
    predictionCol="prediction",
    metricName="f1"
)

ev_precision = MulticlassClassificationEvaluator(
    labelCol="loan_approval",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

ev_recall = MulticlassClassificationEvaluator(
    labelCol="loan_approval",
    predictionCol="prediction",
    metricName="weightedRecall"
)

ev_auc = BinaryClassificationEvaluator(
    labelCol="loan_approval",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# -----------------------------------------------------------------------------
# Evaluate Models
# -----------------------------------------------------------------------------

print("\n")
print("=" * 80)
print("MODEL PERFORMANCE")
print("=" * 80)

results = []

for name, model in models.items():

    start = time.time()

    prediction = model.transform(test)

    prediction_time = time.time() - start

    accuracy = ev_acc.evaluate(prediction)
    precision = ev_precision.evaluate(prediction)
    recall = ev_recall.evaluate(prediction)
    f1 = ev_f1.evaluate(prediction)
    auc = ev_auc.evaluate(prediction)

    results.append({

        "Model": name,
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(auc, 4),
        "Prediction Time (s)": round(prediction_time, 2)

    })

# -----------------------------------------------------------------------------
# Performance Table
# -----------------------------------------------------------------------------

metrics = (
    pd.DataFrame(results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

metrics.index = metrics.index + 1

print("\n")
print("=" * 80)
print("FINAL PERFORMANCE COMPARISON")
print("=" * 80)

print(metrics)

# -----------------------------------------------------------------------------
# Export Results
# -----------------------------------------------------------------------------

metrics.to_csv(
    f"{OUTPUT}/task5_metrics.csv",
    index=False
)

print("\nMetrics exported successfully ✓")

print("=" * 80)
print("COMMIT 1 COMPLETED")
print("=" * 80)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 5 Commit 2: Confusion Matrix and Detailed Model Evaluation
# feat(task5): confusion matrices and classification metrics
#
# Coursework Requirement:
# Analyse classifier predictions using confusion matrices and
# export detailed evaluation results.
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import col

print("="*80)
print("COMMIT 2 : CONFUSION MATRIX & DETAILED EVALUATION")
print("="*80)

# -----------------------------------------------------------------------------
# Evaluate Each Model
# -----------------------------------------------------------------------------

for name, model in models.items():

    print("\n" + "="*80)
    print(name)
    print("="*80)

    prediction = model.transform(test)

    # -------------------------------------------------------------------------
    # Confusion Matrix
    # -------------------------------------------------------------------------

    cm = (
        prediction
        .groupBy("loan_approval", "prediction")
        .count()
        .orderBy("loan_approval", "prediction")
    )

    print("\nConfusion Matrix")

    cm.show()

    cm.toPandas().to_csv(

        f"{OUTPUT}/confusion_matrix_{name.lower().replace(' ','_')}.csv",

        index=False

    )

    # -------------------------------------------------------------------------
    # Evaluation Metrics
    # -------------------------------------------------------------------------

    accuracy = ev_acc.evaluate(prediction)

    precision = ev_precision.evaluate(prediction)

    recall = ev_recall.evaluate(prediction)

    f1 = ev_f1.evaluate(prediction)

    auc = ev_auc.evaluate(prediction)

    print("\nMetrics")

    print(f"Accuracy  : {accuracy:.4f}")

    print(f"Precision : {precision:.4f}")

    print(f"Recall    : {recall:.4f}")

    print(f"F1-Score  : {f1:.4f}")

    print(f"ROC-AUC   : {auc:.4f}")

# -----------------------------------------------------------------------------
# Export Summary Again
# -----------------------------------------------------------------------------

metrics.to_csv(

    f"{OUTPUT}/task5_metrics.csv",

    index=False

)

print("\n")
print("="*80)
print("CONFUSION MATRICES EXPORTED")
print("="*80)

print("Commit 2 Completed ✓")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 3 Commit 3: Perturbation Analysis and SHAP Interpretation
# feat(task5): perturbation analysis and SHAP explainability
#
# Coursework Requirement:
# Evaluate model robustness using perturbation analysis and generate
# SHAP explanations for the best-performing model.
# ──────────────────────────────────────────────────────────────────────────────

import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "shap"]
)

import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier

print("="*80)
print("COMMIT 3 : PERTURBATION ANALYSIS & SHAP")
print("="*80)

# -----------------------------------------------------------------------------
# Baseline Performance (Best Model)
# -----------------------------------------------------------------------------

best_model = models["Gradient Boosted Trees"]

baseline_prediction = best_model.transform(test)

baseline_auc = ev_auc.evaluate(baseline_prediction)

print(f"Baseline ROC-AUC : {baseline_auc:.4f}")

# -----------------------------------------------------------------------------
# Perturbation Analysis
# -----------------------------------------------------------------------------

print("\n")
print("="*80)
print("PERTURBATION ANALYSIS")
print("="*80)

sample_pdf = (
    test
    .select("features", "loan_approval")
    .limit(500)
    .toPandas()
)

X = np.vstack(sample_pdf["features"].apply(lambda x: x.toArray()))
y = sample_pdf["loan_approval"].values

noise_levels = [0.01, 0.03, 0.05]

perturb_results = []

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=7006
)

rf.fit(X, y)

baseline_acc = rf.score(X, y)

for noise in noise_levels:

    np.random.seed(7006)

    X_noisy = X + np.random.normal(
        0,
        noise,
        X.shape
    )

    acc = rf.score(X_noisy, y)

    perturb_results.append({

        "Noise Level": noise,

        "Accuracy": round(acc, 4),

        "Accuracy Drop": round(
            baseline_acc - acc,
            4
        )

    })

perturb_df = pd.DataFrame(
    perturb_results
)

print(perturb_df)

perturb_df.to_csv(
    f"{OUTPUT}/task5_perturbation.csv",
    index=False
)

# -----------------------------------------------------------------------------
# SHAP Analysis
# -----------------------------------------------------------------------------

print("\n")
print("="*80)
print("SHAP ANALYSIS")
print("="*80)

explainer = shap.TreeExplainer(rf)

shap_values = explainer.shap_values(X)

# ------------------------------------------------------------------
# Handle different SHAP versions
# ------------------------------------------------------------------

if isinstance(shap_values, list):
    # Older SHAP versions
    shap_plot = shap_values[1]

elif isinstance(shap_values, np.ndarray):

    if shap_values.ndim == 3:
        shap_plot = shap_values[:, :, 1]
    else:
        shap_plot = shap_values

else:
    shap_plot = shap_values.values

# ------------------------------------------------------------------
# SHAP Summary Plot
# ------------------------------------------------------------------

plt.figure(figsize=(10,6))

shap.summary_plot(
    shap_plot,
    X,
    show=False
)

plt.tight_layout()

plt.savefig(
    f"{OUTPUT}/task5_shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ------------------------------------------------------------------
# SHAP Importance
# ------------------------------------------------------------------

importance = np.abs(shap_plot).mean(axis=0)

feature_names = [
    f"Feature_{i}"
    for i in range(X.shape[1])
]

shap_df = pd.DataFrame({

    "Feature": feature_names,

    "Mean_SHAP": importance

})

shap_df = shap_df.sort_values(
    "Mean_SHAP",
    ascending=False
)

print(shap_df.head(20))

shap_df.to_csv(
    f"{OUTPUT}/task5_shap.csv",
    index=False
)

# -----------------------------------------------------------------------------
# Clean Up
# -----------------------------------------------------------------------------

test.unpersist()

spark.stop()

print("\nSpark Session Stopped ✓")

print("="*80)
print("TASK 5 COMPLETED SUCCESSFULLY")
print("="*80)